# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# Q1 — revenue per order, then totals
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total revenue: ${total_revenue:,.2f} across {len(df)} orders")
print(f"Total units sold: {total_units}")
# Food fair pulled $8,520 in sales moving 783 units.

Total revenue: $8,520.00 across 400 orders
Total units sold: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# Q2 — revenue by category, highest to lowest, with share of total
by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).to_frame()
by_category['share_%'] = (by_category['revenue'] / by_category['revenue'].sum() * 100).round(1)
by_category
# Food alone is ~50% of revenue; RainGear is the tail at ~11%.

,revenue,share_%
category,,
Food,4293.0,50.4
Merch,1771.5,20.8
Drink,1554.0,18.2
RainGear,901.5,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# Q3 — vendor with highest average order revenue (with order counts)
by_vendor = (
    df.groupby('vendor_id')
      .agg(avg_revenue=('revenue', 'mean'), orders=('revenue', 'size'))
      .sort_values('avg_revenue', ascending=False)
)
by_vendor.round(2)
# V-01 has the top average order (~$22.60) on 94 orders, but all four vendors sit within ~$2.

,avg_revenue,orders
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# Q4 — Merch share of total revenue
merch_share = df.loc[df['category'] == 'Merch', 'revenue'].sum() / df['revenue'].sum() * 100
print(f"Merch is {merch_share:.1f}% of total revenue.")

Merch is 20.8% of total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# left join, validated many-to-one
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

# prove nothing changed
assert len(joined) == len(df), 'row count changed'
assert abs(joined['revenue'].sum() - df['revenue'].sum()) < 0.01, 'revenue changed'
print(f"rows: {len(df)} -> {len(joined)}   revenue: ${df['revenue'].sum():,.2f} -> ${joined['revenue'].sum():,.2f}")

# find the vendor id in orders but not in the lookup
unmatched = sorted(joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique())
print(f"unmatched vendor id(s): {unmatched}")

# keep the rows, label the gap instead of dropping (dropping would change the totals above)
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown (V-18)')

rows: 400 -> 400   revenue: $8,520.00 -> $8,520.00
unmatched vendor id(s): ['V-18']


**The unmatched vendor, and what I did about it:** V-18 shows up in the orders but isn't in the lookup, so its name comes back as NaN after the left join. I kept those rows and labeled them `Unknown (V-18)` instead of dropping them, since V-18 is actually the biggest vendor by revenue ($2,349, about 27.6% of the total) and dropping it would break every downstream number.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# Q6 — pivot: vendors down the side, categories across, revenue in cells, with totals
pivot = df.pivot_table(
    index='vendor_id', columns='category', values='revenue',
    aggfunc='sum', margins=True, margins_name='Total',
)
pivot.round(2)

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

**a)** Food is carrying the whole event at $4,293, just over half of all revenue, while RainGear only pulled $901.50 across 46 orders. I'd tell vendors to stock a lot more food and drink and treat rain gear as a small side bet, only bringing much of it when the forecast actually calls for it.

**b)** The Q3 "V-01 has the highest average order" answer is the least trustworthy. All four vendors land within about two dollars of each other ($22.60 down to $20.31), so that ranking is basically noise and would likely flip under a different seed.